In [1]:
!pip install pdfplumber python-docx

In [2]:
import pdfplumber
import re
import json
import os

print("Libraries loaded")

Libraries loaded


In [3]:
def extract_text_from_pdf(path):
    try:
        text = ""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip()
    except Exception as e:
        return ""

TECHNICAL_SKILLS = [
    "Python", "Java", "JavaScript", "C++", "C", "SQL", "HTML", "CSS",
    "React", "Node.js", "FastAPI", "Django", "Flask", "MongoDB",
    "PostgreSQL", "MySQL", "AWS", "Azure", "Docker", "Kubernetes",
    "Git", "GitHub", "TypeScript", "Machine Learning", "Deep Learning",
    "AI", "Artificial Intelligence", "Data Science", "TensorFlow",
    "PyTorch", "NumPy", "Pandas", "LangChain", "REST API", "Linux"
]

def extract_skills(text):
    text_lower = text.lower()
    return [s for s in TECHNICAL_SKILLS if s.lower() in text_lower]

def calculate_ats_score(text, skills):
    score = 0
    has_email = bool(re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text))
    has_phone = bool(re.search(r'\d{10}', text))
    if has_email: score += 10
    if has_phone: score += 10

    skill_count = len(skills)
    if skill_count >= 8: score += 30
    elif skill_count >= 5: score += 20
    elif skill_count >= 1: score += 10

    section_keywords = ["experience", "education", "project", "skill"]
    found = [kw for kw in section_keywords if kw in text.lower()]
    score += min(len(found) * 7, 30)

    word_count = len(text.split())
    if word_count >= 150: score += 20
    elif word_count >= 80: score += 10

    return min(score, 100)

JOB_ROLE_SKILLS = {
    "AI Engineer": ["python", "machine learning", "deep learning", "tensorflow", "pytorch", "docker", "fastapi"],
    "Backend Developer": ["python", "java", "sql", "fastapi", "django", "flask", "docker", "git"],
    "Full Stack Developer": ["javascript", "html", "css", "react", "node.js", "sql", "git"]
}

def skill_gap(your_skills, target_role):
    required = set(s.lower() for s in JOB_ROLE_SKILLS.get(target_role, []))
    yours = set(s.lower() for s in your_skills)
    matched = yours & required
    missing = required - yours
    match_pct = round((len(matched) / len(required)) * 100, 1) if required else 0
    return {"matched": sorted(matched), "missing": sorted(missing), "match_pct": match_pct}

def run_pipeline(resume_path, target_role):
    text = extract_text_from_pdf(resume_path)
    if not text:
        return {"error": f"Could not parse {resume_path}"}
    skills = extract_skills(text)
    ats = calculate_ats_score(text, skills)
    gap = skill_gap(skills, target_role)
    return {
        "resume_file": os.path.basename(resume_path),
        "word_count": len(text.split()),
        "skills_found": skills,
        "skill_count": len(skills),
        "ats_score": ats,
        "target_role": target_role,
        "skill_match_pct": gap["match_pct"],
        "matched_skills": gap["matched"],
        "missing_skills": gap["missing"]
    }

print("Pipeline functions defined")

Pipeline functions defined


In [5]:
result_1 = run_pipeline("../sample_resume/Resume.pdf", "AI Engineer")
print("=== RESUME 1 RESULT ===")
print(json.dumps(result_1, indent=2))

=== RESUME 1 RESULT ===
{
  "resume_file": "Resume.pdf",
  "word_count": 307,
  "skills_found": [
    "Python",
    "Java",
    "JavaScript",
    "C++",
    "C",
    "SQL",
    "HTML",
    "CSS",
    "Flask",
    "Git",
    "Machine Learning",
    "AI",
    "Artificial Intelligence",
    "Data Science",
    "TensorFlow",
    "PyTorch",
    "NumPy",
    "Pandas"
  ],
  "skill_count": 18,
  "ats_score": 98,
  "target_role": "AI Engineer",
  "skill_match_pct": 57.1,
  "matched_skills": [
    "machine learning",
    "python",
    "pytorch",
    "tensorflow"
  ],
  "missing_skills": [
    "deep learning",
    "docker",
    "fastapi"
  ]
}


In [7]:
result_2 = run_pipeline("../sample_resume/Ankush Rana Official Resume.pdf", "Backend Developer")
print("=== RESUME 2 RESULT ===")
print(json.dumps(result_2, indent=2))

=== RESUME 2 RESULT ===
{
  "resume_file": "Ankush Rana Official Resume.pdf",
  "word_count": 466,
  "skills_found": [
    "Python",
    "Java",
    "C++",
    "C",
    "SQL",
    "HTML",
    "CSS",
    "React",
    "Node.js",
    "Flask",
    "MongoDB",
    "MySQL",
    "Git",
    "GitHub",
    "Machine Learning",
    "AI",
    "Artificial Intelligence",
    "NumPy",
    "Pandas",
    "REST API"
  ],
  "skill_count": 20,
  "ats_score": 98,
  "target_role": "Backend Developer",
  "skill_match_pct": 62.5,
  "matched_skills": [
    "flask",
    "git",
    "java",
    "python",
    "sql"
  ],
  "missing_skills": [
    "django",
    "docker",
    "fastapi"
  ]
}


In [8]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Resume": result_1["resume_file"],
        "Target Role": result_1["target_role"],
        "ATS Score": result_1["ats_score"],
        "Skills Found": result_1["skill_count"],
        "Role Match %": result_1["skill_match_pct"]
    },
    {
        "Resume": result_2["resume_file"],
        "Target Role": result_2["target_role"],
        "ATS Score": result_2["ats_score"],
        "Skills Found": result_2["skill_count"],
        "Role Match %": result_2["skill_match_pct"]
    }
])

print("Multi-Resume Pipeline Test: SUCCESS")
print("\nComparison Table:")
print(comparison.to_string(index=False))

Multi-Resume Pipeline Test: SUCCESS

Comparison Table:
                         Resume       Target Role  ATS Score  Skills Found  Role Match %
                     Resume.pdf       AI Engineer         98            18          57.1
Ankush Rana Official Resume.pdf Backend Developer         98            20          62.5


In [9]:
with open("../outputs/multi_resume_test.json", "w", encoding="utf-8") as f:
    json.dump({"resume_1": result_1, "resume_2": result_2}, f, indent=2)

print("\nMulti-resume test saved to ../outputs/multi_resume_test.json")
print("Notebook 12 (Multi-Resume Robustness Test) — COMPLETE")
print("\nThis proves the pipeline generalizes across different resumes and target roles, not hardcoded to one candidate.")


Multi-resume test saved to ../outputs/multi_resume_test.json
Notebook 12 (Multi-Resume Robustness Test) — COMPLETE

This proves the pipeline generalizes across different resumes and target roles, not hardcoded to one candidate.
